In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
xl = pd.ExcelFile("Data.xlsx")

ff_raw = xl.parse("Fama-French Factor")
mf_raw = xl.parse("Mutual Fund")
sb_raw = xl.parse("Smart Beta")
hf_raw = xl.parse("Hedge Fund Index")

print("Sheets loaded:", xl.sheet_names)
print(f"FF Factors     : {ff_raw.shape[0]} rows")
print(f"Mutual Funds   : {mf_raw.shape[0]} rows")
print(f"Smart Beta     : {sb_raw.shape[0]} rows")
print(f"Hedge Funds    : {hf_raw.shape[0]} rows")

Sheets loaded: ['Fama-French Factor', 'Mutual Fund', 'Smart Beta', 'Hedge Fund Index']
FF Factors     : 738 rows
Mutual Funds   : 300 rows
Smart Beta     : 288 rows
Hedge Funds    : 420 rows


In [3]:
# ── Fama-French factors ────────────────────────────────────────────────────────
# FF factors are stored in percent — divide by 100 to convert to decimal
# to match fund returns which are already in decimal format
ff = ff_raw.rename(columns={"Date": "date"}).dropna(subset=["date"])
ff["date"] = ff["date"].astype(int)
ff[["mktrf","smb","hml","RMW","CMA","umd","rf"]] /= 100
ff = ff.set_index("date")

# ── Fund sheets ────────────────────────────────────────────────────────────────
# For each sheet: standardize date format, replace string 'nan' with np.nan,
# coerce to numeric, filter to required sample period, subtract RF for excess returns
def clean_fund_sheet(raw_df, start_yyyymm, end_yyyymm, ff_df):
    df = raw_df.rename(columns={"Date": "date"}).copy()
    df["date"] = df["date"].astype(int)
    df = df.set_index("date")
    df = df.replace("nan", np.nan).apply(pd.to_numeric, errors="coerce")
    df = df.loc[(df.index >= start_yyyymm) & (df.index <= end_yyyymm)]
    excess = df.subtract(ff_df.loc[df.index, "rf"], axis=0)
    return excess.dropna(axis=1, how="all")

mutual_funds = clean_fund_sheet(mf_raw, 200101, 202412, ff)
smart_beta   = clean_fund_sheet(sb_raw, 201001, 202412, ff)
hedge_funds  = clean_fund_sheet(hf_raw, 199101, 202412, ff)

print("=== Post-Cleaning Summary ===")
for label, df in [("Mutual Funds", mutual_funds),
                  ("Smart Beta",   smart_beta),
                  ("Hedge Funds",  hedge_funds)]:
    print(f"  {label}: {df.shape[0]} obs × {df.shape[1]} funds | "
          f"{df.index[0]} → {df.index[-1]} | NaNs: {df.isna().any().any()}")

=== Post-Cleaning Summary ===
  Mutual Funds: 288 obs × 10 funds | 200101 → 202412 | NaNs: False
  Smart Beta: 180 obs × 10 funds | 201001 → 202412 | NaNs: False
  Hedge Funds: 408 obs × 10 funds | 199101 → 202412 | NaNs: False


In [4]:
# Scan all funds for monthly returns exceeding ±50%
# No diversified fund should realistically move more than this in a single month
print("=== Outlier Scan (|return| > 0.50) ===\n")
outliers_found = False

for label, df in [("Mutual Funds", mutual_funds),
                  ("Smart Beta",   smart_beta),
                  ("Hedge Funds",  hedge_funds)]:
    for col in df.columns:
        bad = df[col][df[col].abs() > 0.50].dropna()
        if not bad.empty:
            outliers_found = True
            print(f"  [{label}] {col}: {len(bad)} outlier(s)")
            for date, val in bad.items():
                print(f"    Date {date}: return = {val:.6f}")

if not outliers_found:
    print("  No outliers detected.")

=== Outlier Scan (|return| > 0.50) ===

  [Mutual Funds] INPIX: 1 outlier(s)
    Date 200102: return = -0.508751
  [Mutual Funds] FSPHX: 2 outlier(s)
    Date 201508: return = 8.270287
    Date 201509: return = -0.910750


In [5]:
# Two outliers flagged: FSPHX (201508, 201509) and INPIX (200102)
# Investigate each before deciding whether to fix or retain

# ── FSPHX Aug-Sep 2015 ────────────────────────────────────────────────────────
print("=== FSPHX Investigation (201508, 201509) ===")
print(f"  201508 return : {mutual_funds.loc[201508, 'FSPHX']:.6f}")
print(f"  201509 return : {mutual_funds.loc[201509, 'FSPHX']:.6f}")
print(f"  Market (MKTRF) 201508: {ff.loc[201508, 'mktrf']:.6f}")
print(f"  Market (MKTRF) 201509: {ff.loc[201509, 'mktrf']:.6f}")
print(f"  Verdict: Aug value of 8.27 is clearly a price level stored instead of")
print(f"           a return. Sep value of -0.91 is the correction artifact.")
print(f"           ACTION: replace both with Yahoo Finance verified returns.\n")

# ── INPIX Feb 2000 ────────────────────────────────────────────────────────────
print("=== INPIX Investigation (200102) ===")
mktrf_200102 = ff.loc[200102, "mktrf"]
print(f"  INPIX return        : {mutual_funds.loc[200102, 'INPIX']:.6f}")
print(f"  Market (MKTRF)      : {mktrf_200102:.6f}")
print(f"  SLMCX (tech fund)   : {mutual_funds.loc[200102, 'SLMCX']:.6f}")
print(f"  FSELX (semis fund)  : {mutual_funds.loc[200102, 'FSELX']:.6f}")
print(f"  Implied leverage vs broad market: {mutual_funds.loc[200102, 'INPIX'] / mktrf_200102:.2f}x")
print(f"  Verdict: broad market down 10%, but internet sector down ~25%")
print(f"           (FSELX -30%, SLMCX -21%). At 2x leverage on internet index,")
print(f"           -50% is economically consistent. ACTION: retained as-is.")

=== FSPHX Investigation (201508, 201509) ===
  201508 return : 8.270287
  201509 return : -0.910750
  Market (MKTRF) 201508: -0.060400
  Market (MKTRF) 201509: -0.030700
  Verdict: Aug value of 8.27 is clearly a price level stored instead of
           a return. Sep value of -0.91 is the correction artifact.
           ACTION: replace both with Yahoo Finance verified returns.

=== INPIX Investigation (200102) ===
  INPIX return        : -0.508751
  Market (MKTRF)      : -0.100500
  SLMCX (tech fund)   : -0.205961
  FSELX (semis fund)  : -0.297978
  Implied leverage vs broad market: 5.06x
  Verdict: broad market down 10%, but internet sector down ~25%
           (FSELX -30%, SLMCX -21%). At 2x leverage on internet index,
           -50% is economically consistent. ACTION: retained as-is.


In [6]:
# ── Fix FSPHX only — correct monthly returns sourced from Yahoo Finance ────────
rf_201508 = ff.loc[201508, "rf"]
rf_201509 = ff.loc[201509, "rf"]
mutual_funds.loc[201508, "FSPHX"] = -0.084686 - rf_201508
mutual_funds.loc[201509, "FSPHX"] = -0.096076 - rf_201509

print("=== Fixes Applied ===")
print(f"  FSPHX 201508 → {mutual_funds.loc[201508, 'FSPHX']:.6f}")
print(f"  FSPHX 201509 → {mutual_funds.loc[201509, 'FSPHX']:.6f}")

# ── Final validation ───────────────────────────────────────────────────────────
print("\n=== Final Validation ===")
all_clean = True
for label, df in [("Mutual Funds", mutual_funds),
                  ("Smart Beta",   smart_beta),
                  ("Hedge Funds",  hedge_funds)]:
    remaining = sum(1 for col in df.columns if df[col].abs().max() > 0.50 
                    and col != "INPIX")  # INPIX retained intentionally
    has_nan = df.isna().any().any()
    print(f"  {label}: {df.shape[0]} obs × {df.shape[1]} funds | "
          f"NaNs: {has_nan} | Remaining anomalies: {remaining}")
    if remaining > 0 or has_nan:
        all_clean = False

print(f"\n  Data status: {'✓ Clean — ready for analysis' if all_clean else '✗ Issues remain'}")

FACTORS = ["mktrf", "smb", "hml", "umd", "RMW", "CMA"]

=== Fixes Applied ===
  FSPHX 201508 → -0.084686
  FSPHX 201509 → -0.096076

=== Final Validation ===
  Mutual Funds: 288 obs × 10 funds | NaNs: False | Remaining anomalies: 0
  Smart Beta: 180 obs × 10 funds | NaNs: False | Remaining anomalies: 0
  Hedge Funds: 408 obs × 10 funds | NaNs: False | Remaining anomalies: 0

  Data status: ✓ Clean — ready for analysis


In [7]:
# ── Regression engine ──────────────────────────────────────────────────────────
def run_factor_regression(excess_series, ff_df, factors=FACTORS):
    y = excess_series.dropna()
    X = sm.add_constant(ff_df.loc[y.index, factors])
    model = sm.OLS(y, X).fit()
    result = {}
    result[("Alpha", "coef")]  = model.params["const"]
    result[("Alpha", "tstat")] = model.tvalues["const"]
    for f in factors:
        result[(f, "coef")]  = model.params[f]
        result[(f, "tstat")] = model.tvalues[f]
    result["r_squared"] = model.rsquared
    result["n_obs"]     = int(model.nobs)
    return result

def run_all_regressions(fund_df, ff_df):
    return pd.DataFrame(
        {ticker: run_factor_regression(fund_df[ticker], ff_df)
         for ticker in fund_df.columns}
    ).T

# ── Display engine ─────────────────────────────────────────────────────────────
def format_results_table(results_df, label):
    print(f"\n{'='*85}")
    print(f"  {label}")
    print(f"{'='*85}")
    col_order = ["Alpha"] + FACTORS
    print(f"{'Ticker':<12}" + "".join(f"{c:>10}" for c in col_order) + f"{'R²':>8}  {'N':>5}")
    print("-" * 85)
    for ticker, row in results_df.iterrows():
        coef_line  = f"{ticker:<12}"
        tstat_line = f"{'':12}"
        for col in col_order:
            coef  = row[(col, "coef")]
            tstat = row[(col, "tstat")]
            flag  = " *" if (col == "Alpha" and tstat > 2) else "  "
            coef_line  += f"{coef:>8.4f}{flag}"
            tstat_line += f"({tstat:>6.2f})  "
        coef_line += f"{row['r_squared']:>8.3f}  {int(row['n_obs']):>5}"
        print(coef_line)
        print(tstat_line)
        print()
    sig = results_df[results_df[("Alpha", "tstat")] > 2].index.tolist()
    print(f"  ★ Significant positive alpha (t > 2): {sig if sig else 'None'}")
    print(f"{'='*85}")

# ── Run all three asset classes ────────────────────────────────────────────────
results_mf = run_all_regressions(mutual_funds, ff)
results_sb = run_all_regressions(smart_beta,   ff)
results_hf = run_all_regressions(hedge_funds,  ff)

In [8]:
format_results_table(results_mf, "MUTUAL FUNDS — 200101 to 202412")


  MUTUAL FUNDS — 200101 to 202412
Ticker           Alpha     mktrf       smb       hml       umd       RMW       CMA      R²      N
-------------------------------------------------------------------------------------
FSMEX         0.0038    0.7543    0.2503   -0.2548    0.1589   -0.0838    0.0660     0.477    288
            (  1.66)  ( 13.19)  (  2.78)  ( -2.71)  (  3.05)  ( -0.76)  (  0.48)  

FSELX         0.0066 *  1.2684    0.2863   -0.5477   -0.3164   -0.4696   -0.1092     0.708    288
            (  2.17)  ( 16.66)  (  2.39)  ( -4.38)  ( -4.56)  ( -3.18)  ( -0.60)  

INPIX         0.0072 *  1.7535    0.0187   -0.7969   -0.2100   -1.4743   -0.4763     0.830    288
            (  2.32)  ( 22.58)  (  0.15)  ( -6.24)  ( -2.97)  ( -9.79)  ( -2.57)  

SLMCX         0.0051 *  1.0357    0.2232   -0.3228   -0.1624   -0.2473   -0.2582     0.780    288
            (  2.65)  ( 21.49)  (  2.94)  ( -4.07)  ( -3.70)  ( -2.65)  ( -2.24)  

CSEIX         0.0007    0.8608    0.2852    0.1862   

In [9]:
format_results_table(results_sb, "SMART BETA ETFs — 201001 to 202412")


  SMART BETA ETFs — 201001 to 202412
Ticker           Alpha     mktrf       smb       hml       umd       RMW       CMA      R²      N
-------------------------------------------------------------------------------------
EZM          -0.0011    1.0305    0.5144    0.3556   -0.0528    0.1836   -0.1558     0.955    180
            ( -1.12)  ( 44.31)  ( 12.14)  (  8.52)  ( -1.80)  (  3.50)  ( -2.54)  

FPX           0.0005    1.0858    0.2369   -0.1643    0.0722   -0.4952   -0.1387     0.892    180
            (  0.36)  ( 30.78)  (  3.69)  ( -2.60)  (  1.63)  ( -6.22)  ( -1.49)  

DLN          -0.0012    0.8715   -0.0958    0.1198   -0.0074    0.2015    0.2482     0.951    180
            ( -1.71)  ( 52.59)  ( -3.17)  (  4.03)  ( -0.35)  (  5.39)  (  5.68)  

PSL          -0.0007    0.7585    0.1995   -0.0187    0.2620    0.2084    0.1217     0.688    180
            ( -0.40)  ( 17.69)  (  2.55)  ( -0.24)  (  4.86)  (  2.15)  (  1.08)  

RSP          -0.0013    1.0054    0.1073    0.1743

In [10]:
format_results_table(results_hf, "HEDGE FUND INDICES — 199101 to 202412")


  HEDGE FUND INDICES — 199101 to 202412
Ticker           Alpha     mktrf       smb       hml       umd       RMW       CMA      R²      N
-------------------------------------------------------------------------------------
HFRIDSI       0.0041 *  0.2187    0.1650    0.1374    0.0052   -0.0577   -0.0341     0.481    408
            (  5.69)  ( 12.19)  (  6.61)  (  4.57)  (  0.33)  ( -1.80)  ( -0.77)  

HFRIMAI       0.0026 *  0.1386    0.0923    0.0768    0.0106    0.0037   -0.0036     0.414    408
            (  5.64)  ( 11.87)  (  5.68)  (  3.93)  (  1.05)  (  0.18)  ( -0.13)  

HFRIEMNI      0.0011 *  0.0940    0.0332    0.0517    0.0826    0.0340   -0.0105     0.340    408
            (  3.17)  ( 10.53)  (  2.68)  (  3.46)  ( 10.67)  (  2.13)  ( -0.48)  

HFRIENHI      0.0027 *  0.5660    0.2271   -0.1175    0.0497   -0.0748   -0.0256     0.798    408
            (  3.49)  ( 29.51)  (  8.51)  ( -3.65)  (  2.99)  ( -2.18)  ( -0.54)  

HFRIEM        0.0025    0.4986    0.1671    0.0

In [11]:
#Part 2
from scipy.optimize import minimize

# ── Time period definitions ────────────────────────────────────────────────────
IS_START, IS_END   = 201001, 201912   # in-sample
OOS_START, OOS_END = 202001, 202412   # out-of-sample

def get_period(df, start, end):
    return df.loc[(df.index >= start) & (df.index <= end)]

# ── Annualization ──────────────────────────────────────────────────────────────
def annualize_stats(weights, returns_df):
    """Annualized mean, vol, Sharpe. `returns_df` must be excess returns (as in this notebook)."""
    port_ret = returns_df @ weights
    mean_a   = port_ret.mean() * 12
    std_a    = port_ret.std()  * np.sqrt(12)
    sharpe   = mean_a / std_a
    return mean_a, std_a, sharpe

def get_factor_exposures(weights, returns_df, ff_df, factors=FACTORS):
    """Regress portfolio return on factors, return beta vector."""
    port_ret = returns_df @ weights
    X = sm.add_constant(ff_df.loc[returns_df.index, factors])
    model = sm.OLS(port_ret, X).fit()
    return [model.params[f] for f in factors]

# ── GMV — Global Minimum Variance ─────────────────────────────────────────────
def gmv(cov, long_only=False):
    n = cov.shape[0]
    w0 = np.ones(n) / n
    constraints = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
    bounds = [(0, None)] * n if long_only else [(None, None)] * n
    result = minimize(
        fun=lambda w: w @ cov @ w,
        x0=w0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"ftol": 1e-12, "maxiter": 1000}
    )
    return result.x

# ── MVP — Maximum Sharpe Ratio ─────────────────────────────────────────────────
# mu = monthly mean *excess* return per asset (RF already removed in clean_fund_sheet).
def mvp(mu, cov, long_only=False):
    n = cov.shape[0]
    w0 = np.ones(n) / n
    constraints = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
    bounds = [(0, None)] * n if long_only else [(None, None)] * n
    def neg_sharpe(w):
        ret = w @ mu  # portfolio excess mean — do not subtract rf again
        std = np.sqrt(w @ cov @ w)
        return -ret / std if std > 1e-10 else 1e10
    result = minimize(
        fun=neg_sharpe,
        x0=w0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"ftol": 1e-12, "maxiter": 1000}
    )
    return result.x

# ── MVP + Tracking Error constraint ───────────────────────────────────────────
def mvp_tracking_error(mu, cov, max_te_annual=0.05):
    n = cov.shape[0]
    w0 = np.ones(n) / n
    w_bench = np.ones(n) / n          # equal-weighted benchmark
    max_te_monthly = max_te_annual / np.sqrt(12)
    def neg_sharpe(w):
        ret = w @ mu
        std = np.sqrt(w @ cov @ w)
        return -ret / std if std > 1e-10 else 1e10
    def te_constraint(w):
        diff = w - w_bench
        te = np.sqrt(diff @ cov @ diff)
        return max_te_monthly - te      # monthly TE ≤ 5%/√12  ⇔  annual TE ≤ 5%
    constraints = [
        {"type": "eq",   "fun": lambda w: w.sum() - 1},
        {"type": "ineq", "fun": te_constraint}
    ]
    result = minimize(
        fun=neg_sharpe,
        x0=w0,
        method="SLSQP",
        bounds=None,
        constraints=constraints,
        options={"ftol": 1e-12, "maxiter": 1000}
    )
    return result.x

# ── MVP + Factor Neutral ───────────────────────────────────────────────────────
def mvp_factor_neutral(mu, cov, beta_matrix):
    """
    beta_matrix: (n_funds x n_factors) array of factor loadings from Part I
    Constraint: beta_matrix.T @ w = 0 for all factors
    Weight limit: |w_i| <= 2.0
    """
    n = cov.shape[0]
    w0 = np.ones(n) / n
    def neg_sharpe(w):
        ret = w @ mu
        std = np.sqrt(w @ cov @ w)
        return -ret / std if std > 1e-10 else 1e10
    constraints = [
        {"type": "eq", "fun": lambda w: w.sum() - 1},
        *[{"type": "eq", "fun": lambda w, k=k: beta_matrix[:, k] @ w}
          for k in range(beta_matrix.shape[1])]
    ]
    bounds = [(-2.0, 2.0)] * n
    result = minimize(
        fun=neg_sharpe,
        x0=w0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"ftol": 1e-12, "maxiter": 1000}
    )
    return result.x

In [12]:
def run_optimization(fund_df, ff_df, label, verbose=False):
    """
    Full optimization pipeline for one asset class.
    Returns weight vectors for each strategy. Set verbose=True to print a
    compact IS/OOS table here (otherwise use display_optimization_table below).
    """
    # ── Split data ─────────────────────────────────────────────────────────────
    is_data  = get_period(fund_df, IS_START, IS_END)
    oos_data = get_period(fund_df, OOS_START, OOS_END)

    # ── Estimate inputs from in-sample data only ───────────────────────────────
    mu       = is_data.mean()           # monthly mean *excess* returns
    cov      = is_data.cov()            # monthly covariance (excess returns)

    # ── Extract beta matrix for factor-neutral constraint ─────────────────────
    # Run regression per fund on in-sample data, collect factor loadings
    betas = []
    for ticker in fund_df.columns:
        y = is_data[ticker].dropna()
        X = sm.add_constant(ff_df.loc[y.index, FACTORS])
        model = sm.OLS(y, X).fit()
        betas.append([model.params[f] for f in FACTORS])
    beta_matrix = np.array(betas)   # shape: (n_funds, n_factors)

    mu_arr  = mu.values
    cov_arr = cov.values

    # ── Solve all 6 portfolios ─────────────────────────────────────────────────
    portfolios = {
        "Long-Only GMV":   gmv(cov_arr, long_only=True),
        "Long-Only MVP":   mvp(mu_arr, cov_arr, long_only=True),
        "Long-Short GMV":  gmv(cov_arr, long_only=False),
        "Long-Short MVP":  mvp(mu_arr, cov_arr, long_only=False),
        "TE<=5% MVP":      mvp_tracking_error(mu_arr, cov_arr),
        "Factor Neutral":  mvp_factor_neutral(mu_arr, cov_arr, beta_matrix),
    }

    # IS/OOS tables: see display_optimization_table below.
    if verbose:
        print(f"\n{'='*75}\n  {label}\n{'='*75}")
        hdr = (f"{'Portfolio':<18} {'Period':<12} {'Mean':>8} {'Vol':>8} "
               f"{'Sharpe':>8} {'mktrf':>7} {'smb':>7} {'hml':>7} "
               f"{'umd':>7} {'RMW':>7} {'CMA':>7}")
        print(hdr + "\n" + "-" * 95)
        for port_name, weights in portfolios.items():
            for period_label, period_data in [("In-Sample", is_data),
                                              ("OOS",       oos_data)]:
                mean_a, std_a, sharpe = annualize_stats(weights, period_data)
                factor_exp = get_factor_exposures(weights, period_data, ff_df)
                print(f"{port_name:<18} {period_label:<12} "
                      f"{mean_a:>8.4f} {std_a:>8.4f} {sharpe:>8.4f} "
                      + "".join(f"{b:>7.3f}" for b in factor_exp))
            print()
    else:
        print(f"Solved 6 portfolios for {label} (run display cells below for full tables).")

    return portfolios

# ── Run for all three asset classes ───────────────────────────────────────────
port_mf = run_optimization(mutual_funds, ff, "MUTUAL FUNDS")
port_sb = run_optimization(smart_beta,   ff, "SMART BETA ETFs")
port_hf = run_optimization(hedge_funds,  ff, "HEDGE FUND INDICES")

Solved 6 portfolios for MUTUAL FUNDS (run display cells below for full tables).
Solved 6 portfolios for SMART BETA ETFs (run display cells below for full tables).
Solved 6 portfolios for HEDGE FUND INDICES (run display cells below for full tables).


In [13]:
def display_optimization_table(fund_df, ff_df, portfolios, label):
    """
    Formats results into a structured IS/OOS table with annualized stats and factor exposures.
    in-sample block then out-of-sample block, annualized stats + factor exposures.
    """
    is_data  = get_period(fund_df, IS_START, IS_END)
    oos_data = get_period(fund_df, OOS_START, OOS_END)

    port_order = ["Long-Only GMV", "Long-Only MVP",
                  "Long-Short GMV", "Long-Short MVP",
                  "TE<=5% MVP", "Factor Neutral"]

    header = (f"{'Portfolio':<18} {'Mean':>8} {'Vol':>8} {'Sharpe':>8} "
              f"{'mktrf':>8} {'smb':>8} {'hml':>8} "
              f"{'umd':>8} {'RMW':>8} {'CMA':>8}")
    divider = "─" * len(header)

    for period_label, period_data in [("IN-SAMPLE  (201001–201912)", is_data),
                                       ("OUT-OF-SAMPLE (202001–202412)", oos_data)]:
        print(f"\n{'='*90}")
        print(f"  {label}  |  {period_label}")
        print(f"{'='*90}")
        print(header)
        print(divider)

        for port_name in port_order:
            w = portfolios[port_name]
            mean_a, std_a, sharpe = annualize_stats(w, period_data)
            fexp = get_factor_exposures(w, period_data, ff_df)
            print(f"{port_name:<18} "
                  f"{mean_a:>8.4f} {std_a:>8.4f} {sharpe:>8.4f} "
                  + "".join(f"{b:>8.4f}" for b in fexp))
        print(divider)

# Run the per–asset-class cells below to print tables (avoids duplicate output
# if this definition cell is re-run after those cells).

In [14]:
display_optimization_table(mutual_funds, ff, port_mf, "MUTUAL FUNDS")


  MUTUAL FUNDS  |  IN-SAMPLE  (201001–201912)
Portfolio              Mean      Vol   Sharpe    mktrf      smb      hml      umd      RMW      CMA
───────────────────────────────────────────────────────────────────────────────────────────────────
Long-Only GMV        0.1310   0.1191   1.1000   0.9190 -0.1263 -0.1466  0.0724  0.0278 -0.0373
Long-Only MVP        0.1618   0.1361   1.1890   0.9600 -0.0310 -0.3544  0.0159 -0.2106 -0.1746
Long-Short GMV       0.0664   0.0842   0.7878   0.5240 -0.0904 -0.1373  0.0290  0.2893  0.2636
Long-Short MVP       0.1738   0.1363   1.2749   0.9304 -0.0160 -0.4804 -0.0273 -0.1649  0.3123
TE<=5% MVP           0.1738   0.1379   1.2605   0.9602 -0.0026 -0.4327 -0.0121 -0.2092  0.1338
Factor Neutral       0.0620   0.4817   0.1288  -0.0000 -0.0000  0.0000 -0.0000  0.0000  0.0000
───────────────────────────────────────────────────────────────────────────────────────────────────

  MUTUAL FUNDS  |  OUT-OF-SAMPLE (202001–202412)
Portfolio              Mean      

In [15]:
display_optimization_table(smart_beta, ff, port_sb, "SMART BETA ETFs")


  SMART BETA ETFs  |  IN-SAMPLE  (201001–201912)
Portfolio              Mean      Vol   Sharpe    mktrf      smb      hml      umd      RMW      CMA
───────────────────────────────────────────────────────────────────────────────────────────────────
Long-Only GMV        0.0286   0.0442   0.6478   0.1561  0.0527 -0.0404 -0.0129 -0.1124  0.0104
Long-Only MVP        0.1148   0.0937   1.2257   0.7220  0.0106 -0.1541  0.1244  0.1734  0.1706
Long-Short GMV       0.0318   0.0365   0.8723   0.1059 -0.0543 -0.0536  0.0207  0.0505  0.0785
Long-Short MVP       0.1491   0.0790   1.8881   0.4498 -0.1991 -0.2927  0.1974 -0.0304  0.3240
TE<=5% MVP           0.1541   0.0993   1.5513   0.7673 -0.0659 -0.2066  0.1189  0.0284  0.2217
Factor Neutral       0.1028   0.1082   0.9504   0.0000  0.0000  0.0000 -0.0000  0.0000  0.0000
───────────────────────────────────────────────────────────────────────────────────────────────────

  SMART BETA ETFs  |  OUT-OF-SAMPLE (202001–202412)
Portfolio              Mean

In [16]:
display_optimization_table(hedge_funds, ff, port_hf, "HEDGE FUND INDICES")


  HEDGE FUND INDICES  |  IN-SAMPLE  (201001–201912)
Portfolio              Mean      Vol   Sharpe    mktrf      smb      hml      umd      RMW      CMA
───────────────────────────────────────────────────────────────────────────────────────────────────
Long-Only GMV        0.0260   0.0192   1.3515   0.1151 -0.0070  0.0066  0.0195 -0.0326 -0.0284
Long-Only MVP        0.0290   0.0203   1.4318   0.1078 -0.0010 -0.0029  0.0033 -0.0435 -0.0369
Long-Short GMV       0.0131   0.0122   1.0676   0.0135 -0.0298  0.0068  0.0247 -0.0561 -0.0004
Long-Short MVP       0.0395   0.0213   1.8518   0.0646  0.0081  0.0407  0.0221  0.0047  0.0165
TE<=5% MVP           0.0395   0.0213   1.8518   0.0646  0.0081  0.0407  0.0221  0.0047  0.0165
Factor Neutral       0.0331   0.0221   1.4985  -0.0000  0.0000 -0.0000 -0.0000 -0.0000  0.0000
───────────────────────────────────────────────────────────────────────────────────────────────────

  HEDGE FUND INDICES  |  OUT-OF-SAMPLE (202001–202412)
Portfolio            